# Columbina Pretrain — Daily Colab Session

Run this notebook top to bottom once per training session (roughly the rhythm from the plan:
30-60 min if the TEST cell is doing real work for the first time in a while, 2-4 hours for the
TRAIN cell, then disconnect). It is meant to be run **many times over many months** — every
run after the first one resumes from whatever the previous run left on Google Drive, picking
up exactly where it left off.

**What "the plan" means here**: `C:\Users\alanl\.claude\plans\alright-now-lets-start-purring-volcano.md`
in the repo this notebook came from. Read that first if anything below is confusing — this
notebook is the executable half of that plan, not a standalone thing.

**Status of what this notebook actually trains on right now**: Phase 2 (the real
FineWeb-Edu/OpenWebText/dialogue corpus) hasn't been built yet, so until `CORPUS_PATH` below
actually exists on Drive, this notebook generates a small synthetic placeholder corpus instead
— enough to prove every piece of the pipeline (config, resume, checkpointing, the daily
rhythm) works correctly on a real Colab GPU, without claiming to be real training yet. Swap
`CONFIG_NAME` to `"406m"` and point `CORPUS_PATH` at the real corpus once Phase 2/3 land.

## 1. Setup

Installs the one package not already in the Colab image, mounts Drive (this is where
checkpoints and the corpus persist — Colab's local disk is wiped every session), and pulls
the latest `pretrain/` code from GitHub. First run clones; every run after that just pulls, so
a local code fix always reaches the next session automatically.

In [ ]:
!pip install -q tiktoken

import os
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Columbina_Pretrain"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL = "https://github.com/YENOSven/gpt-bina.git"
CODE_DIR = "/content/gpt-bina"

if os.path.exists(CODE_DIR):
    subprocess.run(["git", "-C", CODE_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, CODE_DIR], check=True)

sys.path.insert(0, f"{CODE_DIR}/pretrain")
print(f"code ready at {CODE_DIR}, Drive mounted at {DRIVE_ROOT}")


## 2. Session config

Change `STAGE`/`CONFIG_NAME` here as the project progresses (`pretrain` -> `sft` -> `dpo`,
`124m` -> `406m`) — checked into git via this notebook, not hand-typed differently each
session, so every session uses the same target on purpose, not by accident.

`TOTAL_STEPS` is the schedule's target for the **entire run**, not this session — see the
Phase-0 writeup in `pretrain/README.md` for why that distinction matters (an earlier bug had
this wrong and silently corrupted the LR schedule on every simulated "resume"). It's a
placeholder until Phase 3 sizes the real run from the real corpus's token count.

In [ ]:
STAGE = "pretrain"           # pretrain | sft | dpo
CONFIG_NAME = "124m"         # 124m for now (free-tier-friendly); 406m once Phase 2/3 are ready

STAGE_DIR = f"{DRIVE_ROOT}/{STAGE}"
os.makedirs(f"{STAGE_DIR}/milestones", exist_ok=True)
LATEST_CKPT = f"{STAGE_DIR}/latest.pt"
CORPUS_PATH = f"{DRIVE_ROOT}/corpus.bin"   # Phase 2 writes the real corpus here

TOTAL_STEPS = 2000           # placeholder -- real value computed once Phase 3 sizes the run
WARMUP_STEPS = 100
MICRO_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4

CHECKPOINT_EVERY_MINUTES = 20    # within the plan's stated 15-30 minute range
MILESTONE_EVERY_STEPS = 2000
TRAIN_SESSION_MINUTES = 150      # 2.5 hours -- inside the plan's 2-4hr range with safety margin
STOP_MARGIN_MINUTES = 10         # stop this much early so the final checkpoint write is never rushed

print(f"stage={STAGE} config={CONFIG_NAME} checkpoint={LATEST_CKPT}")


## 3. Corpus check

If Phase 2's real corpus isn't on Drive yet, generate a small synthetic placeholder so this
notebook is fully testable today. This never overwrites a real corpus that's already there.

In [ ]:
import numpy as np

if not os.path.exists(CORPUS_PATH):
    print("no corpus on Drive yet (Phase 2 hasn't run) -- generating a synthetic placeholder")
    rng = np.random.default_rng(0)
    tokens = rng.integers(0, 50257, size=5_000_000, dtype=np.uint16)
    tokens.tofile(CORPUS_PATH)
print(f"corpus: {os.path.getsize(CORPUS_PATH):,} bytes at {CORPUS_PATH}")


## 4. TEST cell — must pass before TRAIN is allowed to spend real GPU time

Two checks, both against this session's **real** Drive mount (not a local-disk stand-in):

1. `scripts/verify_resume_local.py` — the exact Phase-0 proof (bit-identical resume across
   separate subprocesses), pointed at a throwaway scratch path on Drive instead of local disk.
   This is what actually re-proves cross-session resume: run this notebook today, then again
   on a different day, and both days independently re-verify the mechanism against Drive's
   real (occasionally laggy) write-propagation — not just a same-process approximation of it.
2. A short real training run (10 steps) against the actual corpus, checked for a finite,
   sensibly-behaved loss.

**Fails loudly on purpose** (raises, doesn't warn) — if this cell fails, stop and fix the
pipeline before running TRAIN. That's what protects the compute budget.

In [ ]:
import json
import math
import subprocess
import sys
import time

t0 = time.time()

print("[TEST 1/2] resume-and-continue mechanism, against real Drive I/O...")
proof_dir = f"{DRIVE_ROOT}/_resume_proof"
result = subprocess.run(
    [sys.executable, "scripts/verify_resume_local.py", "--scratch-dir", proof_dir],
    cwd=f"{CODE_DIR}/pretrain", capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("TEST FAILED: resume-and-continue proof did not pass -- do not run TRAIN")

print("\n[TEST 2/2] short real training run against the actual corpus...")
test_ckpt = f"{DRIVE_ROOT}/_test_run.pt"
test_log = f"{DRIVE_ROOT}/_test_run_losses.json"
result = subprocess.run(
    [sys.executable, "-m", "columbina_pretrain.train",
     "--config", CONFIG_NAME, "--corpus-path", CORPUS_PATH,
     "--total-steps", "10", "--warmup-steps", "3",
     "--micro-batch-size", "2", "--grad-accum-steps", "2",
     "--checkpoint-every", "10", "--save-path", test_ckpt, "--log-path", test_log],
    cwd=f"{CODE_DIR}/pretrain", capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("TEST FAILED: short training run errored -- do not run TRAIN")

losses = json.loads(open(test_log).read())
if not all(math.isfinite(loss_value) for loss_value in losses):
    raise RuntimeError(f"TEST FAILED: non-finite loss in {losses} -- do not run TRAIN")
print(f"loss: {losses[0]:.3f} -> {losses[-1]:.3f}")

print(f"\nALL TESTS PASSED in {(time.time() - t0) / 60:.1f} min -- safe to run TRAIN below.")


## 5. Resume detection for the real run

Separate from the TEST cell's throwaway proof checkpoint above — this loads (or starts fresh)
the **actual** `latest.pt` for `STAGE`, which is what TRAIN continues from.

In [ ]:
import torch

from columbina_pretrain import checkpoint as ckpt
from columbina_pretrain.model import build_model
from columbina_pretrain.train import CONFIGS, make_optimizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

model = build_model(CONFIGS[CONFIG_NAME], device=device)
optimizer = make_optimizer(model)

resume_state = ckpt.try_resume(LATEST_CKPT, model, optimizer, map_location=device)
if resume_state is None:
    print(f"no checkpoint at {LATEST_CKPT} -- this is the first session for stage='{STAGE}', starting fresh")
    start_step = 0
    global_token_cursor = 0
    wall_clock_seconds_trained = 0.0
else:
    start_step = resume_state["step"] + 1
    global_token_cursor = resume_state["global_token_cursor"]
    wall_clock_seconds_trained = resume_state["wall_clock_seconds_trained"]
    hours_so_far = wall_clock_seconds_trained / 3600
    print(f"resuming stage='{STAGE}' from step {start_step}/{TOTAL_STEPS} "
          f"({hours_so_far:.1f} GPU-hours trained so far)")


## 6. TRAIN cell — the real 2-4 hour budget

Checkpoints `latest.pt` every `CHECKPOINT_EVERY_MINUTES` by wall clock (not a fixed step
count — step time varies with whatever GPU Colab hands out this session), writes a rotating
milestone every `MILESTONE_EVERY_STEPS`, and stops itself `STOP_MARGIN_MINUTES` before the
session budget runs out so the final save is always clean rather than racing a disconnect.

In [ ]:
import time

import torch
import torch.nn.functional as F

from columbina_pretrain.train import get_batch_sequential, load_corpus, lr_multiplier, PEAK_LR, MAX_GRAD_NORM

corpus = load_corpus(CORPUS_PATH)
block_size = CONFIGS[CONFIG_NAME]["max_seq_len"]
vocab_size = CONFIGS[CONFIG_NAME]["vocab_size"]

session_deadline = time.time() + (TRAIN_SESSION_MINUTES - STOP_MARGIN_MINUTES) * 60
last_checkpoint_time = time.time()
step = start_step
tokens_this_session = 0

while step < TOTAL_STEPS and time.time() < session_deadline:
    lr = PEAK_LR * lr_multiplier(step, WARMUP_STEPS, TOTAL_STEPS)
    for group in optimizer.param_groups:
        group["lr"] = lr

    optimizer.zero_grad()
    for _ in range(GRAD_ACCUM_STEPS):
        x, y, global_token_cursor = get_batch_sequential(corpus, block_size, MICRO_BATCH_SIZE, global_token_cursor)
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type="cuda" if device == "cuda" else "cpu", dtype=torch.bfloat16):
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1)) / GRAD_ACCUM_STEPS
        loss.backward()
        tokens_this_session += MICRO_BATCH_SIZE * block_size

    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    optimizer.step()

    if step % 50 == 0:
        print(f"step {step}/{TOTAL_STEPS}  lr {lr:.2e}  loss {loss.item() * GRAD_ACCUM_STEPS:.4f}")

    now = time.time()
    if now - last_checkpoint_time > CHECKPOINT_EVERY_MINUTES * 60 or step == TOTAL_STEPS - 1:
        wall_clock_seconds_trained += now - last_checkpoint_time
        ckpt.save_checkpoint(model, optimizer, step, global_token_cursor, LATEST_CKPT, wall_clock_seconds_trained)
        last_checkpoint_time = now
        print(f"  checkpoint saved at step {step} ({wall_clock_seconds_trained / 3600:.2f} GPU-hours total)")
        if (step + 1) % MILESTONE_EVERY_STEPS == 0:
            milestone_path = f"{STAGE_DIR}/milestones/ckpt_step_{step:06d}.pt"
            ckpt.save_checkpoint(model, optimizer, step, global_token_cursor, milestone_path, wall_clock_seconds_trained)
            print(f"  milestone saved: {milestone_path}")

    step += 1

print(f"\nsession ended at step {step}/{TOTAL_STEPS}, {tokens_this_session:,} tokens trained this session")


## 7. Session summary and disconnect

Prints where things stand, then releases the GPU. `google.colab.runtime.unassign()` avoids
idle billing after the cell finishes — falls back to a manual reminder if that API isn't
available in the current Colab environment.

In [ ]:
sessions_remaining = max(0, TOTAL_STEPS - step) / max(1, step - start_step or 1)
print(f"""
=== SESSION SUMMARY ===
stage: {STAGE}  config: {CONFIG_NAME}
step: {step} / {TOTAL_STEPS}
total GPU-hours trained (all sessions): {wall_clock_seconds_trained / 3600:.2f}
checkpoint: {LATEST_CKPT}
Drive usage: run `!du -sh {DRIVE_ROOT}` if you want an exact number

Safe to disconnect now -- the checkpoint above is a clean, complete save.
""")

try:
    from google.colab import runtime
    runtime.unassign()
except Exception:
    print("Could not auto-disconnect (google.colab.runtime API unavailable in this environment) "
          "-- close this tab / disconnect manually from the Runtime menu.")
